# Lesson 3 — Output Parsers

## Goal

understanding:

- Why output parsing exists.
- Why prompt engineering alone isn't enough
- What an output parser is.
- where output parsers fit in the LangChain pipeline.
- The most common parsers.
- how structures output is used in production AI systems.

In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load the environment
load_dotenv()

True

In [3]:
# Read the model name
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [4]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [5]:
# Create LangChain prompt template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages (
    [
        ("system","You are a senior AI engineer"),
        ("human", "Explain {topic}"),
    ]
)

In [6]:
# Render the template
prompt_value = prompt.invoke (
    {
        "topic":"LangChain"
    }
)

In [7]:
# Invoke the model
response = llm.invoke(prompt_value) 

In [ ]:
# Output parser
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

text = parser.invoke(response)

print (type(text))
print (text)

<class 'langchain_core.messages.base.TextAccessor'>
As a senior AI engineer, I look at **LangChain** not just as a library, but as the **orchestration layer** for modern LLM (Large Language Model) applications. 

At its core, LangChain is an open-source framework designed to simplify the construction of applications that combine LLMs with external sources of computation, data, and APIs.

Here is a comprehensive breakdown of what LangChain is, why it exists, its architecture, and an honest engineering assessment of its pros and cons.

---

### 1. The Core Problem LangChain Solves
Out of the box, an LLM (like GPT-4 or Claude) is a stateless "brain." It has three major limitations:
1. **No Memory:** It doesn't remember past interactions.
2. **No Fresh Data:** It only knows what was in its training data (knowledge cutoff).
3. **No Agency:** It cannot take actions (like querying a database, searching the web, or sending an email) on its own.

**LangChain acts as the "glue."** It connects th

In [9]:
print(text == response.content)

False


In [10]:
print(type(response.content))

print(type(text))

print(repr(response.content)) # repr() returns the official string representation of an object.

print(repr(text))

<class 'list'>
<class 'langchain_core.messages.base.TextAccessor'>
[{'type': 'text', 'text': 'As a senior AI engineer, I look at **LangChain** not just as a library, but as the **orchestration layer** for modern LLM (Large Language Model) applications. \n\nAt its core, LangChain is an open-source framework designed to simplify the construction of applications that combine LLMs with external sources of computation, data, and APIs.\n\nHere is a comprehensive breakdown of what LangChain is, why it exists, its architecture, and an honest engineering assessment of its pros and cons.\n\n---\n\n### 1. The Core Problem LangChain Solves\nOut of the box, an LLM (like GPT-4 or Claude) is a stateless "brain." It has three major limitations:\n1. **No Memory:** It doesn\'t remember past interactions.\n2. **No Fresh Data:** It only knows what was in its training data (knowledge cutoff).\n3. **No Agency:** It cannot take actions (like querying a database, searching the web, or sending an email) on its

In [11]:
print(isinstance(response.content, str)) # isinstance() checks whether an object belongs to a certain class (or one of its subclasses).

print(isinstance(text, str))

False
True


In [12]:
print(dir(text)) # return everything an object knows how to do. 

['__add__', '__call__', '__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mod__', '__module__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__rmod__', '__rmul__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', 'capitalize', 'casefold', 'center', 'count', 'encode', 'endswith', 'expandtabs', 'find', 'format', 'format_map', 'index', 'isalnum', 'isalpha', 'isascii', 'isdecimal', 'isdigit', 'isidentifier', 'islower', 'isnumeric', 'isprintable', 'isspace', 'istitle', 'isupper', 'join', 'ljust', 'lower', 'lstrip', 'maketrans', 'partition', 'removeprefix', 'removesuffix', 'replace', 'rfind', 'rindex', 'rjust', 'rpartition', 'rsplit', 'rstrip', 'split', 'splitlines', 'startswith', 'strip', 'swapcase', 'title', 'translate'

In [13]:
print(response.content)

[{'type': 'text', 'text': 'As a senior AI engineer, I look at **LangChain** not just as a library, but as the **orchestration layer** for modern LLM (Large Language Model) applications. \n\nAt its core, LangChain is an open-source framework designed to simplify the construction of applications that combine LLMs with external sources of computation, data, and APIs.\n\nHere is a comprehensive breakdown of what LangChain is, why it exists, its architecture, and an honest engineering assessment of its pros and cons.\n\n---\n\n### 1. The Core Problem LangChain Solves\nOut of the box, an LLM (like GPT-4 or Claude) is a stateless "brain." It has three major limitations:\n1. **No Memory:** It doesn\'t remember past interactions.\n2. **No Fresh Data:** It only knows what was in its training data (knowledge cutoff).\n3. **No Agency:** It cannot take actions (like querying a database, searching the web, or sending an email) on its own.\n\n**LangChain acts as the "glue."** It connects the LLM to d

In [14]:
# Output parser
from langchain_core.output_parsers import JsonOutputParser
parser = JsonOutputParser() # expects JSON 
parser.get_format_instructions()

'Return a JSON object.'

In [15]:
instructions = parser.get_format_instructions()
print(type(instructions))

<class 'str'>


### Complete chain

``` text
PromptTemplate
      │
      ▼
ChatGoogleGenerativeAI
      │
      ▼
JsonOutputParser
      │
      ▼
Python dict

```

In [28]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template = """
Answer the following question.

{format_instructions}

Question:
{question}
""",
    input_variables= ["question"],
    partial_variables = {
        "format_instructions": parser.get_format_instructions()
    }
)

In [29]:
print(prompt.invoke({"question":"Tell me about Egypt"}).text)


Answer the following question.

Return a JSON object.

Question:
Tell me about Egypt



In [30]:
# Build the prompt

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a helpful assistant.
            {format_instructions}
            """,
        ),
        ("human", "{question}")
    ]
). partial(format_instructions = parser.get_format_instructions()) 
"""
ChatPromptTemplate does not accept partial_variables in the same way that prompt template does
.partial() creates a new prompt where some variables are already filled in
The parser instructions have become part of the prompt itself.
.partial is used so every invocation automatically includes variable. Now the parser instructions have become part of the prompt itself
"""

'\nChatPromptTemplate does not accept partial_variables in the same way that prompt template does\n.partial() creates a new prompt where some variables are already filled in\nThe parser instructions have become part of the prompt itself.\n.partial is used so every invocation automatically includes variable. Now the parser instructions have become part of the prompt itself\n'

In [33]:
# Render the prompt

prompt_value = prompt.invoke(
    {
        "question": "Tell me about Egypt"
    }
)
prompt_value

ChatPromptValue(messages=[SystemMessage(content='\n            You are a helpful assistant.\n            Return a JSON object.\n            ', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about Egypt', additional_kwargs={}, response_metadata={})])

In [ ]:
# Invoke the model
response = llm.invoke(prompt_value)

AIMessage(content=[{'type': 'text', 'text': '{\n  "country": "Egypt",\n  "official_name": "Arab Republic of Egypt",\n  "capital": "Cairo",\n  "location": {\n    "continent": "Africa (transcontinental, spanning northeast Africa and southwest Asia via the Sinai Peninsula)",\n    "borders": [\n      "Libya",\n      "Sudan",\n      "Israel",\n      "Gaza Strip"\n    ],\n    "bodies_of_water": [\n      "Mediterranean Sea",\n      "Red Sea"\n    ]\n  },\n  "demographics": {\n    "population_estimate": "110+ million",\n    "official_language": "Arabic",\n    "primary_religion": "Islam (predominantly Sunni), with a significant Coptic Christian minority"\n  },\n  "economy": {\n    "currency": "Egyptian Pound (EGP)",\n    "key_revenue_sources": [\n      "Tourism",\n      "Suez Canal tolls",\n      "Remittances from Egyptians working abroad",\n      "Agriculture (cotton, rice, fruits)",\n      "Natural gas and petroleum"\n    ]\n  },\n  "geography": {\n    "key_features": [\n      "The Nile River

In [ ]:
print(type(response)) # without the parser
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
[{'type': 'text', 'text': '{\n  "country": "Egypt",\n  "official_name": "Arab Republic of Egypt",\n  "capital": "Cairo",\n  "location": {\n    "continent": "Africa (transcontinental, spanning northeast Africa and southwest Asia via the Sinai Peninsula)",\n    "borders": [\n      "Libya",\n      "Sudan",\n      "Israel",\n      "Gaza Strip"\n    ],\n    "bodies_of_water": [\n      "Mediterranean Sea",\n      "Red Sea"\n    ]\n  },\n  "demographics": {\n    "population_estimate": "110+ million",\n    "official_language": "Arabic",\n    "primary_religion": "Islam (predominantly Sunni), with a significant Coptic Christian minority"\n  },\n  "economy": {\n    "currency": "Egyptian Pound (EGP)",\n    "key_revenue_sources": [\n      "Tourism",\n      "Suez Canal tolls",\n      "Remittances from Egyptians working abroad",\n      "Agriculture (cotton, rice, fruits)",\n      "Natural gas and petroleum"\n    ]\n  },\n  "geography": {\n    "key_featur

In [35]:
# Use the parser
result = parser.invoke(response)

print(type(result))
print(result)

<class 'dict'>
{'country': 'Egypt', 'official_name': 'Arab Republic of Egypt', 'capital': 'Cairo', 'location': {'continent': 'Africa (transcontinental, spanning northeast Africa and southwest Asia via the Sinai Peninsula)', 'borders': ['Libya', 'Sudan', 'Israel', 'Gaza Strip'], 'bodies_of_water': ['Mediterranean Sea', 'Red Sea']}, 'demographics': {'population_estimate': '110+ million', 'official_language': 'Arabic', 'primary_religion': 'Islam (predominantly Sunni), with a significant Coptic Christian minority'}, 'economy': {'currency': 'Egyptian Pound (EGP)', 'key_revenue_sources': ['Tourism', 'Suez Canal tolls', 'Remittances from Egyptians working abroad', 'Agriculture (cotton, rice, fruits)', 'Natural gas and petroleum']}, 'geography': {'key_features': ['The Nile River (the lifeblood of the country)', 'The Sahara Desert (Western and Eastern Deserts)', 'The Sinai Peninsula', 'The Suez Canal (a vital global shipping lane)']}, 'history': {'ancient_egypt': "One of the world's earliest an

### The data flow

```text
User Input
     │
     ▼
ChatPromptTemplate
     │
     ▼
PromptValue
     │
     ▼
Gemini
     │
     ▼
AIMessage
(content is JSON text)
     │
     ▼
JsonOutputParser
     │
     ▼
Python dict
```